# 데이터 집계와 그룹 연산
## 1. 그룹 연산에 대한 고찰
분리-적용-결합(split-apply-combine)

In [1]:
import pandas as pd
import numpy as np
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                   "key2" : pd.Series([1, 2, 1, 2, 1, None, 1],
                                      dtype="Int64"),
                   "data1" : np.random.standard_normal(7),
                   "data2" : np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.434968,-1.891277
1,a,2,0.047750,1.601602
2,None,1,-0.836024,-2.011687
3,b,2,0.255970,0.367380
4,b,1,-0.985892,-0.883273
5,a,<NA>,-0.329452,1.306848
6,None,1,0.797747,0.145605


In [2]:
grouped = df['data1'].groupby(df['key1'])
grouped

In [3]:
grouped.mean()

key1
a   -0.238890
b   -0.364961
Name: data1, dtype: float64

In [6]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means

key1  key2
a     1      -0.434968
      2       0.047750
b     1      -0.985892
      2       0.255970
Name: data1, dtype: float64

In [7]:
means.unstack()

key2,1,2
key1,,
a,-0.434968,0.04775
b,-0.985892,0.25597


In [8]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]
df["data1"].groupby([states, years]).mean()

CA  2005   -0.140851
    2006   -0.836024
OH  2005   -0.089499
    2006   -0.094073
Name: data1, dtype: float64

In [9]:
df.groupby('key1').mean()

,key2,data1,data2
key1,,,
a,1.5,-0.238890,0.339058
b,1.5,-0.364961,-0.257946


In [10]:
df.groupby('key2').mean(numeric_only=True)

,data1,data2
key2,,
1,-0.364784,-1.160158
2,0.151860,0.984491


In [12]:
df.groupby(['key1', 'key2']).size().unstack()

key2,1,2
key1,,
a,1,1
b,1,1


In [13]:
df.groupby('key1', dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [14]:
df.groupby(['key1','key2'], dropna=False).size().unstack()

key2,1,2,<NA>
key1,,,
a,1.0,1.0,1.0
b,1.0,1.0,NaN
NaN,2.0,NaN,NaN


In [15]:
df.groupby('key1').count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


In [16]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1  key2     data1     data2
0    a     1 -0.434968 -1.891277
1    a     2  0.047750  1.601602
5    a  <NA> -0.329452  1.306848
b
  key1  key2     data1     data2
3    b     2  0.255970  0.367380
4    b     1 -0.985892 -0.883273


In [18]:
for (k1, k2), group in df.groupby(['key1','key2']):
    print((k1,k2))
    print(group)

('a', np.int64(1))
  key1  key2     data1     data2
0    a     1 -0.434968 -1.891277
('a', np.int64(2))
  key1  key2    data1     data2
1    a     2  0.04775  1.601602
('b', np.int64(1))
  key1  key2     data1     data2
4    b     1 -0.985892 -0.883273
('b', np.int64(2))
  key1  key2    data1    data2
3    b     2  0.25597  0.36738


In [19]:
pieces = {name : group for name, group in df.groupby('key1')}
pieces['b']

,key1,key2,data1,data2
3,b,2,0.255970,0.367380
4,b,1,-0.985892,-0.883273


In [20]:
df.groupby(['key1','key2'])['data2'].mean()

key1  key2
a     1      -1.891277
      2       1.601602
b     1      -0.883273
      2       0.367380
Name: data2, dtype: float64

In [21]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

,a,b,c,d,e
Joe,-0.140296,-2.399200,0.194071,0.933130,0.296181
Steve,-0.459254,-0.008308,-0.591399,-0.403168,-2.339361
Wanda,-0.703227,NaN,NaN,0.612824,1.088820
Jill,-0.680249,-0.265845,-0.383093,0.087571,-0.103428
Trey,0.102286,-0.293070,0.886257,-1.895685,-1.322750


In [22]:
mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f" : "orange"}

In [ ]:
by_column = people.groupby(mapping, axis=1)
by_column.sum()

/var/folders/gr/1r0zsf75231d5gylt0gl0wsm0000gn/T/ipykernel_13264/2922953290.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping, axis=1)


,blue,red
Joe,1.127200,-2.243314
Steve,-0.994567,-2.806923
Wanda,0.612824,0.385593
Jill,-0.295522,-1.049523
Trey,-1.009428,-1.513533


In [25]:
people.groupby(len).sum()

,a,b,c,d,e
3,-0.140296,-2.399200,0.194071,0.933130,0.296181
4,-0.577963,-0.558915,0.503164,-1.808114,-1.426178
5,-1.162481,-0.008308,-0.591399,0.209657,-1.250542


In [26]:
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,-0.140296,-2.399200,0.194071,0.933130,0.296181
4,two,-0.680249,-0.293070,-0.383093,-1.895685,-1.322750
5,one,-0.703227,-0.008308,-0.591399,-0.403168,-2.339361


In [27]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                    [1, 3, 5, 1, 3]],
                                    names=["cty", "tenor"])
hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

cty          US                                JP          
tenor         1         3             5         1         3
0      0.001538 -1.040707  1.385377e-07  0.596008  0.343369
1      0.888045  1.055204 -5.921981e-01  1.551243 -0.206289
2     -1.014027  0.248621 -5.653680e-01 -0.773217 -1.127899
3      0.699048  0.893816  2.072571e-01 -0.558505 -1.343500

In [28]:
hier_df.groupby(level='cty', axis=1).count()

/var/folders/gr/1r0zsf75231d5gylt0gl0wsm0000gn/T/ipykernel_13264/2333815484.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  hier_df.groupby(level='cty', axis=1).count()


cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [29]:
df

,key1,key2,data1,data2
0,a,1,-0.434968,-1.891277
1,a,2,0.047750,1.601602
2,None,1,-0.836024,-2.011687
3,b,2,0.255970,0.367380
4,b,1,-0.985892,-0.883273
5,a,<NA>,-0.329452,1.306848
6,None,1,0.797747,0.145605


In [30]:
grouped = df.groupby('key1')
grouped['data1'].nsmallest(2)

key1   
a     0   -0.434968
      5   -0.329452
b     4   -0.985892
      3    0.255970
Name: data1, dtype: float64

In [31]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,0.482719,3.492879
b,1,1.241862,1.250653


In [32]:
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.238890  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.364961  ...   

                        data2                                          \
           75%      max count      mean       std       min       25%   
key1                                                                    
a    -0.140851  0.04775   3.0  0.339058  1.937141 -1.891277 -0.292215   
b    -0.054496  0.25597   2.0 -0.257946  0.884345 -0.883273 -0.570609   

                                    
           50%       75%       max  
key1                                
a     1.306848  1.454225  1.601602  
b    -0.257946  0.054717  0.367380  

[2 rows x 24 columns]

In [34]:
tips = pd.read_csv('https://github.com/wesm/pydata-book/blob/3rd-edition/examples/tips.csv?raw=True')
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [35]:
tips['tip_pct'] = tips['tip'] / (tips['total_bill'] - tips['tip'])
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.063204
1,10.34,1.66,No,Sun,Dinner,3,0.191244
2,21.01,3.50,No,Sun,Dinner,3,0.199886
3,23.68,3.31,No,Sun,Dinner,2,0.162494
4,24.59,3.61,No,Sun,Dinner,4,0.172069


In [36]:
grouped = tips.groupby(['day','smoker'])
grouped_pct = grouped['tip_pct']
grouped_pct.agg('mean')

day   smoker
Fri   No        0.179740
      Yes       0.216293
Sat   No        0.190412
      Yes       0.179833
Sun   No        0.193617
      Yes       0.322021
Thur  No        0.193424
      Yes       0.198508
Name: tip_pct, dtype: float64

In [37]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.179740  0.039458      0.094263
     Yes     0.216293  0.077530      0.242219
Sat  No      0.190412  0.058626      0.352192
     Yes     0.179833  0.089496      0.446137
Sun  No      0.193617  0.060302      0.274897
     Yes     0.322021  0.538061      2.382107
Thur No      0.193424  0.056065      0.284273
     Yes     0.198508  0.057170      0.219047

In [41]:
grouped_pct.agg([('average', 'mean'), ('stdev', 'std')])

average     stdev
day  smoker                    
Fri  No      0.179740  0.039458
     Yes     0.216293  0.077530
Sat  No      0.190412  0.058626
     Yes     0.179833  0.089496
Sun  No      0.193617  0.060302
     Yes     0.322021  0.538061
Thur No      0.193424  0.056065
     Yes     0.198508  0.057170

In [43]:
functions = ['count', 'mean', 'max']
result = grouped[['tip_pct','total_bill']].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.179740  0.231125          4  18.420000  22.75
     Yes         15  0.216293  0.357737         15  16.813333  40.17
Sat  No          45  0.190412  0.412409         45  19.661778  48.33
     Yes         42  0.179833  0.483092         42  21.276667  50.81
Sun  No          57  0.193617  0.338101         57  20.506667  48.17
     Yes         19  0.322021  2.452381         19  24.120000  45.35
Thur No          45  0.193424  0.362976         45  17.113111  41.19
     Yes         17  0.198508  0.317965         17  19.190588  43.11

In [44]:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.179740  0.231125
     Yes        15  0.216293  0.357737
Sat  No         45  0.190412  0.412409
     Yes        42  0.179833  0.483092
Sun  No         57  0.193617  0.338101
     Yes        19  0.322021  2.452381
Thur No         45  0.193424  0.362976
     Yes        17  0.198508  0.317965

In [46]:
ftuples = [("Average", "mean"), ("Variance", 'var')]
grouped[["tip_pct", "total_bill"]].agg(ftuples)

tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.179740  0.001557  18.420000   25.596333
     Yes     0.216293  0.006011  16.813333   82.562438
Sat  No      0.190412  0.003437  19.661778   79.908965
     Yes     0.179833  0.008010  21.276667  101.387535
Sun  No      0.193617  0.003636  20.506667   66.099980
     Yes     0.322021  0.289509  24.120000  109.046044
Thur No      0.193424  0.003143  17.113111   59.625081
     Yes     0.198508  0.003268  19.190588   69.808518

In [48]:
grouped.agg({"tip" : 'max', "size" : "sum"})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [49]:
grouped.agg({"tip_pct" : ["min", "max", "mean", "std"],
             "size" : "sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.136861  0.231125  0.179740  0.039458    9
     Yes     0.115518  0.357737  0.216293  0.077530   31
Sat  No      0.060217  0.412409  0.190412  0.058626  115
     Yes     0.036955  0.483092  0.179833  0.089496  104
Sun  No      0.063204  0.338101  0.193617  0.060302  167
     Yes     0.070274  2.452381  0.322021  0.538061   49
Thur No      0.078704  0.362976  0.193424  0.056065  112
     Yes     0.098918  0.317965  0.198508  0.057170   40

In [51]:
grouped = tips.groupby(['day','smoker'], as_index=False)
grouped.mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.179740
1,Fri,Yes,16.813333,2.714000,2.066667,0.216293
2,Sat,No,19.661778,3.102889,2.555556,0.190412
3,Sat,Yes,21.276667,2.875476,2.476190,0.179833
4,Sun,No,20.506667,3.167895,2.929825,0.193617
5,Sun,Yes,24.120000,3.516842,2.578947,0.322021
6,Thur,No,17.113111,2.673778,2.488889,0.193424
7,Thur,Yes,19.190588,3.030000,2.352941,0.198508


In [52]:
def top(df, n=5, column="tip_pct"):
    return df.sort_values(column, ascending=False)[:n]
top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,2.452381
178,9.60,4.00,Yes,Sun,Dinner,2,0.714286
67,3.07,1.00,Yes,Sat,Dinner,1,0.483092
232,11.61,3.39,No,Sat,Dinner,2,0.412409
183,23.17,6.50,Yes,Sun,Dinner,4,0.389922
109,14.31,4.00,Yes,Sat,Dinner,2,0.387973


In [55]:
tips.groupby('smoker').apply(top, include_groups = False)

total_bill   tip   day    time  size   tip_pct
smoker                                                    
No     232       11.61  3.39   Sat  Dinner     2  0.412409
       149        7.51  2.00  Thur   Lunch     2  0.362976
       51        10.29  2.60   Sun  Dinner     2  0.338101
       185       20.69  5.00   Sun  Dinner     5  0.318674
       88        24.71  5.85  Thur   Lunch     2  0.310180
Yes    172        7.25  5.15   Sun  Dinner     2  2.452381
       178        9.60  4.00   Sun  Dinner     2  0.714286
       67         3.07  1.00   Sat  Dinner     1  0.483092
       183       23.17  6.50   Sun  Dinner     4  0.389922
       109       14.31  4.00   Sat  Dinner     2  0.387973

In [57]:
tips.groupby(['smoker','day']).apply(top, n=1, column='total_bill', include_groups = False)

total_bill    tip    time  size   tip_pct
smoker day                                                
No     Fri  94        22.75   3.25  Dinner     2  0.166667
       Sat  212       48.33   9.00  Dinner     4  0.228833
       Sun  156       48.17   5.00  Dinner     6  0.115821
       Thur 142       41.19   5.00   Lunch     5  0.138160
Yes    Fri  95        40.17   4.73  Dinner     4  0.133465
       Sat  170       50.81  10.00  Dinner     3  0.245038
       Sun  182       45.35   3.50  Dinner     3  0.083632
       Thur 197       43.11   5.00   Lunch     4  0.131199

In [59]:
result = tips.groupby('smoker')['tip_pct'].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.192237,0.057665,0.060217,0.158622,0.184308,0.227015,0.412409
Yes,93.0,0.218176,0.254295,0.036955,0.119534,0.181818,0.242326,2.452381


In [60]:
result.unstack('smoker')

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.192237
       Yes         0.218176
std    No          0.057665
       Yes         0.254295
min    No          0.060217
       Yes         0.036955
25%    No          0.158622
       Yes         0.119534
50%    No          0.184308
       Yes         0.181818
75%    No          0.227015
       Yes         0.242326
max    No          0.412409
       Yes         2.452381
dtype: float64

In [61]:
tips.groupby('smoker').apply(top)

/var/folders/gr/1r0zsf75231d5gylt0gl0wsm0000gn/T/ipykernel_13264/1695234324.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby('smoker').apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.412409
       149        7.51  2.00     No  Thur   Lunch     2  0.362976
       51        10.29  2.60     No   Sun  Dinner     2  0.338101
       185       20.69  5.00     No   Sun  Dinner     5  0.318674
       88        24.71  5.85     No  Thur   Lunch     2  0.310180
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  2.452381
       178        9.60  4.00    Yes   Sun  Dinner     2  0.714286
       67         3.07  1.00    Yes   Sat  Dinner     1  0.483092
       183       23.17  6.50    Yes   Sun  Dinner     4  0.389922
       109       14.31  4.00    Yes   Sat  Dinner     2  0.387973

In [62]:
tips.groupby('smoker', group_keys=False).apply(top)

/var/folders/gr/1r0zsf75231d5gylt0gl0wsm0000gn/T/ipykernel_13264/1502793219.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby('smoker', group_keys=False).apply(top)


,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.412409
149,7.51,2.00,No,Thur,Lunch,2,0.362976
51,10.29,2.60,No,Sun,Dinner,2,0.338101
185,20.69,5.00,No,Sun,Dinner,5,0.318674
88,24.71,5.85,No,Thur,Lunch,2,0.310180
172,7.25,5.15,Yes,Sun,Dinner,2,2.452381
178,9.60,4.00,Yes,Sun,Dinner,2,0.714286
67,3.07,1.00,Yes,Sat,Dinner,1,0.483092
183,23.17,6.50,Yes,Sun,Dinner,4,0.389922
109,14.31,4.00,Yes,Sat,Dinner,2,0.387973


In [63]:
frame = pd.DataFrame({'data1' : np.random.standard_normal(1000),
                      'data2' : np.random.standard_normal(1000)})

frame.head()

,data1,data2
0,0.641273,0.666028
1,0.025854,0.428208
2,1.106829,-0.067688
3,-0.670351,-1.084359
4,-0.200459,0.418178


In [64]:
quartiles = pd.cut(frame['data1'], 4)
quartiles.head(10)

0      (-0.107, 1.41]
1      (-0.107, 1.41]
2      (-0.107, 1.41]
3    (-1.625, -0.107]
4    (-1.625, -0.107]
5    (-1.625, -0.107]
6      (-0.107, 1.41]
7      (-0.107, 1.41]
8      (-0.107, 1.41]
9    (-1.625, -0.107]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-3.148, -1.625] < (-1.625, -0.107] < (-0.107, 1.41] < (1.41, 2.927]]

In [65]:
def get_stats(group):
    return pd.DataFrame(
        {"min" : group.min(),
         "max" : group.max(),
         'count' : group.count(),
         'mean' : group.mean()
         }
    )
    
grouped = frame.groupby(quartiles)
grouped.apply(get_stats)

/var/folders/gr/1r0zsf75231d5gylt0gl0wsm0000gn/T/ipykernel_13264/745021271.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


min       max  count      mean
data1                                                      
(-3.148, -1.625] data1 -3.142162 -1.627700     67 -2.068150
                 data2 -2.295645  2.629890     67  0.115739
(-1.625, -0.107] data1 -1.617834 -0.108692    418 -0.713353
                 data2 -2.792247  3.515087    418  0.034914
(-0.107, 1.41]   data1 -0.105038  1.408656    437  0.529222
                 data2 -3.278096  2.632313    437 -0.099523
(1.41, 2.927]    data1  1.422817  2.927221     78  1.794561
                 data2 -2.231070  2.007408     78  0.038255

In [66]:
grouped.agg(['min','max','count','mean'])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-3.148, -1.625] -3.142162 -1.627700    67 -2.068150 -2.295645  2.629890   
(-1.625, -0.107] -1.617834 -0.108692   418 -0.713353 -2.792247  3.515087   
(-0.107, 1.41]   -0.105038  1.408656   437  0.529222 -3.278096  2.632313   
(1.41, 2.927]     1.422817  2.927221    78  1.794561 -2.231070  2.007408   

                                  
                 count      mean  
data1                             
(-3.148, -1.625]    67  0.115739  
(-1.625, -0.107]   418  0.034914  
(-0.107, 1.41]     437 -0.099523  
(1.41, 2.927]       78  0.038255

In [68]:
quartiles_samp = pd.qcut(frame['data1'], 4, labels=False)
quartiles_samp.head() 

0    3
1    2
2    3
3    1
4    1
Name: data1, dtype: int64

In [69]:
grouped = frame.groupby(quartiles_samp)

In [70]:
grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -3.142162 -0.718788    250 -1.361967
      data2 -2.792247  2.629890    250  0.080625
1     data1 -0.714672 -0.070709    250 -0.390306
      data2 -3.278096  3.515087    250  0.005381
2     data1 -0.065787  0.627115    250  0.263194
      data2 -2.845975  2.465127    250 -0.103634
3     data1  0.628400  2.927221    250  1.227071
      data2 -2.351674  2.632313    250 -0.055008